# Primary processing: sequencing reads to count matrix (Cell Ranger)

**Status: work in progress.**

The breast-cancer analysis in `01_breast_ddr.ipynb` is a complete single-cell analysis, run from raw
counts through to the biological result. The CELLxGENE atlas it uses was supplied already aligned and
quantified by the original authors, so the one upstream step not performed there is primary
processing: the conversion of raw 10x Genomics sequencing reads into a cell-by-gene count matrix.
This notebook documents that step with Cell Ranger and connects the resulting matrix to the same
downstream pipeline.

## Overview

Primary processing has three parts, after which the count matrix enters the established downstream
pipeline:

1. Obtain 10x Genomics reads (FASTQ files) for a chosen sample.
2. Obtain a reference transcriptome.
3. Run `cellranger count` to align reads and quantify expression, producing a filtered cell-by-gene
   matrix.
4. Load the resulting matrix into Scanpy.
5. Continue with the same downstream pipeline as `01_breast_ddr.ipynb` (quality control,
   normalisation, feature selection, dimensionality reduction, clustering, and annotation).

Parts 1 to 3 require sequencing data, a reference, and substantial compute (Cell Ranger is run from
the shell, not from Python), and are documented here as commands rather than executed inline. Parts 4
and 5 run in this environment once a count matrix is available.

## Step 1: obtain reads (FASTQ)

Reads are retrieved from a public repository (for example SRA or a 10x Genomics public dataset). The
files must follow the Cell Ranger naming convention `SAMPLE_S1_L001_R1_001.fastq.gz` and
`SAMPLE_S1_L001_R2_001.fastq.gz`.

```bash
mkdir -p fastq
# prefetch <SRR_ACCESSION> && fasterq-dump --split-files <SRR_ACCESSION> -O fastq/
```

## Step 2: reference transcriptome

Cell Ranger requires a prebuilt reference. The 10x Genomics human reference (GRCh38) is used here.

```bash
curl -O "https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-GRCh38-2020-A.tar.gz"
tar -xzf refdata-gex-GRCh38-2020-A.tar.gz
```

## Step 3: generate the count matrix (Cell Ranger)

`cellranger count` aligns the reads, calls cells, and quantifies expression. The filtered output is
written to `runs/SAMPLE/outs/filtered_feature_bc_matrix/`.

```bash
cellranger count \
  --id=SAMPLE \
  --fastqs=fastq/ \
  --sample=SAMPLE \
  --transcriptome=refdata-gex-GRCh38-2020-A \
  --output-dir=runs/SAMPLE
```

## Step 4: load the count matrix into Scanpy

The filtered matrix is read directly with `scanpy.read_10x_mtx`, the point at which primary processing
hands off to the downstream pipeline. The cell below runs once a Cell Ranger output directory is
present.

In [ ]:
import scanpy as sc

# Path to the Cell Ranger filtered matrix (set once Step 3 has been run)
MATRIX_DIR = "runs/SAMPLE/outs/filtered_feature_bc_matrix"

# adata = sc.read_10x_mtx(MATRIX_DIR, var_names="gene_symbols", cache=True)
# adata.var_names_make_unique()
# print(adata)

## Step 5: downstream analysis

From this point the analysis is identical to `01_breast_ddr.ipynb`: per-cell quality-control metrics
and distribution-based filtering, normalisation (counts per 10,000 and log1p), selection of highly
variable genes, scaling, PCA, neighbour-graph construction, Leiden clustering, UMAP, and marker-based
annotation. The corresponding sections of `01_breast_ddr.ipynb` are reused, so that the only
difference between the two notebooks is the source of the count matrix.

## Requirements and next steps

- Cell Ranger (10x Genomics) installed and on the shell path.
- A selected public 10x sample with available FASTQ files.
- The GRCh38 reference and sufficient disk and memory for alignment.

Next step: select a public breast-cancer 10x sample, run Steps 1 to 3, and connect the resulting
matrix to the downstream pipeline.